# Tworzenie cech (feature engineering)

**Plan notatnika:**
1. Czym jest feature engineering
2. Cechy z daty i czasu
3. Cechy opóźnione i okno kroczące
4. Dyskretyzacja zmiennej ciągłej
5. Cechy z danych złożonych: listy i tekst
6. Podsumowanie

Dokumentacja: [scikit-learn.org/stable/modules/feature_extraction.html](https://scikit-learn.org/stable/modules/feature_extraction.html)

## 1. Czym jest feature engineering

Jakość modelu zależy od cech (kolumn), na których się uczy. **Feature engineering** to tworzenie nowych cech z istniejących danych tak, by wzorce były dla modelu łatwiejsze do znalezienia. Przykładowo z daty `2024-01-27` nie wynika wprost, że to sobota, a dla sprzedaży w sklepie to ważna informacja.

Nowe cechy tworzymy głównie za pomocą pandas. scikit-learn ma moduł `sklearn.feature_extraction` (głównie dla tekstu) oraz transformery takie jak `KBinsDiscretizer` i `MultiLabelBinarizer`.

In [ ]:
import numpy as np
import pandas as pd
import sklearn

sklearn.__version__

## 2. Cechy z daty i czasu

Do przykładu użyjemy dziennej sprzedaży w sklepie z 10 kolejnych dni. Data jest w indeksie.

In [ ]:
rng = np.random.default_rng(42)

df_sales = pd.DataFrame(
    {'sales': rng.integers(80, 200, size=10)},
    index=pd.date_range('2024-01-26', periods=10, freq='D', name='date'),
)
df_sales

Z indeksu czasowego można odczytać części daty (`.day`, `.month`, `.year`, `.dayofweek`, `.quarter`):

In [ ]:
df_sales['day'] = df_sales.index.day
df_sales['month'] = df_sales.index.month
df_sales['year'] = df_sales.index.year
df_sales['dayofweek'] = df_sales.index.dayofweek        # 0 = poniedziałek, 6 = niedziela
df_sales['is_weekend'] = (df_sales['dayofweek'] >= 5).astype(int)
df_sales

**Cechy cykliczne.** Dzień tygodnia `6` (niedziela) i `0` (poniedziałek) sąsiadują ze sobą, a w danych liczbowych dzieli je odległość 6. Cykliczność oddają funkcje sinus i cosinus:

In [ ]:
df_sales['dow_sin'] = np.sin(2 * np.pi * df_sales['dayofweek'] / 7)
df_sales['dow_cos'] = np.cos(2 * np.pi * df_sales['dayofweek'] / 7)
df_sales[['dayofweek', 'dow_sin', 'dow_cos']].round(2)

## 3. Cechy opóźnione i okno kroczące

W szeregach czasowych bardzo przydatne są wartości z przeszłości:
- `shift(1)` – wartość z poprzedniego dnia (*lag*),
- `rolling(3).mean()` – średnia krocząca z 3 ostatnich dni.

**Uwaga:** średnia z okna zawierającego bieżący dzień „zna” wynik, który mamy przewidzieć. Dlatego najpierw przesuwamy szereg (`shift(1)`), a potem liczymy średnią.

In [ ]:
df_lag = df_sales[['sales']].copy()
df_lag['sales_lag1'] = df_lag['sales'].shift(1)
df_lag['sales_mean3'] = df_lag['sales'].shift(1).rolling(3).mean()
df_lag

Na początku szeregu pojawiają się braki (`NaN`), bo nie ma jeszcze historii. Obsługę braków omówiliśmy w notatniku 02 (`dropna()` lub imputacja).

## 4. Dyskretyzacja zmiennej ciągłej

**Dyskretyzacja** (*binning*) zamienia liczby na przedziały, np. wzrost na kategorie `small`, `medium`, `high`. Pomaga, gdy zależność nie jest liniowa lub gdy chcemy ograniczyć wpływ wartości odstających. Kosztem jest utrata szczegółów.

In [ ]:
df_height = pd.DataFrame({'height': [175.0, 178.5, 185.0, 191.0, 184.5, 183.0, 168.0]})
df_height

`pd.cut()` z liczbą przedziałów dzieli zakres na części o **równej szerokości**:

In [ ]:
pd.cut(df_height['height'], bins=3)

Granice można podać ręcznie, wraz z nazwami przedziałów. Przedziały są domknięte z prawej strony: `(160, 175]` zawiera 175.

In [ ]:
df_height['height_cat'] = pd.cut(
    df_height['height'],
    bins=[160, 175, 180, 195],
    labels=['small', 'medium', 'high'],
)
df_height

`pd.qcut()` dzieli na przedziały o **równej liczbie obserwacji** (kwantyle):

In [ ]:
pd.qcut(df_height['height'], q=3, labels=['small', 'medium', 'high'])

Odpowiednik w scikit-learn to `KBinsDiscretizer`, który można dołączyć do `ColumnTransformer`:

In [ ]:
from sklearn.preprocessing import KBinsDiscretizer

kbins = KBinsDiscretizer(n_bins=3, encode='ordinal', strategy='quantile')
kbins.fit_transform(df_height[['height']])

Otrzymane kategorie możemy zakodować one-hot (patrz notatnik 01):

In [ ]:
pd.get_dummies(df_height, columns=['height_cat'], prefix='height', dtype=int)

## 5. Cechy z danych złożonych: listy i tekst

### Listy

Kolumna może zawierać listy (np. znane języki).

In [ ]:
df_lang = pd.DataFrame({'lang': [['PL', 'ENG'], ['GER', 'ENG', 'PL', 'FRA'], ['RUS']]})
df_lang

In [ ]:
df_lang['lang_number'] = df_lang['lang'].apply(len)                           # liczba języków
df_lang['PL_flag'] = df_lang['lang'].apply(lambda x: 1 if 'PL' in x else 0)   # czy zna polski
df_lang

Gdy chcemy mieć kolumnę dla **każdego** języka, używamy `MultiLabelBinarizer` (odpowiednik one-hot dla list):

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
pd.DataFrame(mlb.fit_transform(df_lang['lang']), columns=mlb.classes_, index=df_lang.index)

### Tekst w kolumnie – `str.split()`

In [ ]:
df_web = pd.DataFrame({'website': ['wp.pl', 'onet.pl', 'google.com', 'bbc.co.uk']})
df_web

`n=1` oznacza podział tylko na pierwszej kropce. Dzięki temu `co.uk` pozostaje w jednej kolumnie.

In [ ]:
df_web[['portal', 'extension']] = df_web['website'].str.split('.', n=1, expand=True)
df_web

### Tekst jako liczby – `CountVectorizer`

Model nie rozumie zdań, więc trzeba je zamienić na liczby. `CountVectorizer` buduje słownik wszystkich słów i dla każdego dokumentu **liczy wystąpienia** każdego słowa (*bag of words*).

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

docs = ['Python jest świetny', 'Python i uczenie maszynowe', 'uczenie jest świetne']

vectorizer = CountVectorizer()
counts = vectorizer.fit_transform(docs)

pd.DataFrame(counts.toarray(), columns=vectorizer.get_feature_names_out())

Wariant `TfidfVectorizer` zmniejsza wagę słów występujących w wielu dokumentach (mało informacyjnych). Zaawansowane metody pracy z tekstem (m.in. modele językowe) poznamy w rozdziale 5.

## 6. Podsumowanie

| Rodzaj danych | Technika | Narzędzie |
|---|---|---|
| Data i czas | dzień, miesiąc, dzień tygodnia, weekend, sin/cos | `.index.day`, `.dayofweek`, `np.sin` |
| Szeregi czasowe | wartości opóźnione, średnia krocząca | `shift()`, `rolling()` |
| Liczby ciągłe | przedziały | `pd.cut`, `pd.qcut`, `KBinsDiscretizer` |
| Listy kategorii | flagi, liczność | `apply`, `MultiLabelBinarizer` |
| Tekst w kolumnie | podział na części | `str.split()` |
| Dłuższy tekst | liczenie słów | `CountVectorizer`, `TfidfVectorizer` |

**Najważniejsze zasady:**
1. Twórz cechy, które mają sens dla problemu (np. weekend dla sprzedaży).
2. W szeregach czasowych nie używaj informacji z przyszłości (`shift` przed `rolling`).
3. Nowe cechy mogą dawać braki i wymagają dalszego przygotowania (notatniki 01 i 02).